# BuildFlowMatch (waft-a2, dav2) - demo fine-tuning in Colab

Pasi: clone repo -> instalare dependinte -> checkpoint + encoder dav2 (din Google Drive) -> incarcare set propriu de date (poze + ground truth) -> fine-tuning cateva pasi -> vizualizare flux optic inainte/dupa.

## 1. Clonare repo
Inlocuieste `REPO_URL` cu URL-ul repo-ului tau (cel pe care l-ai facut push cu acest folder minimal).

In [ ]:
REPO_URL = "https://github.com/rosemariestoica13/BuildFlowMatch.git"

!git clone $REPO_URL buildflowmatch
%cd buildflowmatch

## 2. Instalare dependinte

In [ ]:
!pip install -q -r requirements.txt

## 3. Checkpoint-uri (BuildFlowMatch + encoder dav2) din Google Drive
Se descarca automat din folderul tau Google Drive:
https://drive.google.com/drive/folders/1jg476sKB1r9xMA2ecoMvWVLHDtpl5HMi

Ca sa mearga descarcarea automata (fara login), folderul trebuie setat pe
**"Oricine are linkul" -> Vizualizator** (Anyone with the link - Viewer). Daca nu merge,
foloseste celula de fallback cu upload manual de mai jos.

In [ ]:
!pip install -q gdown

import os, shutil, gdown

FOLDER_URL = "https://drive.google.com/drive/folders/1jg476sKB1r9xMA2ecoMvWVLHDtpl5HMi"

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('depth-anything-ckpts', exist_ok=True)

gdown.download_folder(url=FOLDER_URL, output='drive_ckpts', quiet=False, use_cookies=False)

for fname in os.listdir('drive_ckpts'):
    src = os.path.join('drive_ckpts', fname)
    if 'depth_anything' in fname.lower():
        shutil.move(src, os.path.join('depth-anything-ckpts', fname))
    else:
        shutil.move(src, os.path.join('checkpoints', 'sintel-gm-final.pth'))

print('checkpoints/:', os.listdir('checkpoints'))
print('depth-anything-ckpts/:', os.listdir('depth-anything-ckpts'))

In [ ]:
# Ruleaza doar daca descarcarea automata de mai sus a esuat (ex. folderul nu e public).
# Selecteaza ambele fisiere .pth deodata (checkpoint-ul BuildFlowMatch + depth_anything_v2_vits.pth).
from google.colab import files
uploaded = files.upload()
for fname in uploaded:
    if 'depth_anything' in fname.lower():
        os.rename(fname, os.path.join('depth-anything-ckpts', 'depth_anything_v2_vits.pth'))
    else:
        os.rename(fname, os.path.join('checkpoints', 'sintel-gm-final.pth'))

## 5. Setul tau de date (poze + ground truth)
Incarca o arhiva `.zip` cu structura:
```
image1/000000.png ...
image2/000000.png ...
flow/000000.flo ...
```
(vezi README.md pentru detalii despre formatul `.flo`).

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('data/custom', exist_ok=True)
uploaded = files.upload()  # selecteaza arhiva .zip cu image1/ image2/ flow/
for fname in uploaded:
    with zipfile.ZipFile(fname) as z:
        z.extractall('data/custom')

print(os.listdir('data/custom'))

## 6. Fine-tuning
Ruleaza cativa pasi de antrenare pornind de la checkpoint, pe setul tau de date.

In [ ]:
!python finetune_demo.py \
    --cfg config/a2/dav2/sintel-gm.json \
    --ckpt checkpoints/sintel-gm-final.pth \
    --data_dir data/custom \
    --steps 100 \
    --out_ckpt checkpoints/finetuned.pth

## 7. Vizualizare inainte / dupa fine-tuning

In [ ]:
import matplotlib.pyplot as plt
import cv2

before = cv2.cvtColor(cv2.imread('demo_out/flow_before.jpg'), cv2.COLOR_BGR2RGB)
after = cv2.cvtColor(cv2.imread('demo_out/flow_after.jpg'), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(before); ax[0].set_title('Flux optic - inainte de fine-tuning'); ax[0].axis('off')
ax[1].imshow(after); ax[1].set_title('Flux optic - dupa fine-tuning'); ax[1].axis('off')
plt.show()